# Phase 1: Exploratory Data Analysis (EDA) - Loan Default Prediction

**Objective:** Uncover underlying patterns, anomalies, missingness, and structural relationships in the loan portfolio to inform feature engineering and modeling.

> **Mentor Rule Reminder:** We exclusively use `data/train.csv` during exploration and model training to avoid data leakage.

### Step 1.1: Environment Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Pandas formatting & visualization styling
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment configured successfully.")

### Step 1.2: Ingestion & Dimensionality Check

In [ ]:
TRAIN_PATH = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
print(f"Loading: {TRAIN_PATH.resolve()}")

df_train = pd.read_csv(TRAIN_PATH)
print(f"\nDataset Dimensions: {df_train.shape[0]:,} rows x {df_train.shape[1]} columns")
df_train.head()

### Step 1.3: Target Distribution & Imbalance Analysis

In credit risk, assessing the base default rate is critical. A naive classifier predicting non-default (0) on all loans would yield deceptively high accuracy while causing severe credit losses.

In [ ]:
target_counts = df_train['loan_default'].value_counts()
target_pct = df_train['loan_default'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct
})
print("--- Target Class Distribution ---")
display(summary_target)

# Visualize target balance
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Non-Default (0)', 'Default (1)'], target_counts.values, color=['#2b5c8f', '#d95f02'], width=0.5)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:,}\n({height/len(df_train):.1%})',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Number of Loans')
ax.set_title('Loan Default Target Balance (Class Distribution)', fontsize=13, pad=12)
ax.set_ylim(0, target_counts.max() * 1.18)
plt.tight_layout()
plt.show()

### Step 1.4: Missing Value Audit (Explicit & Implicit)

We inspect both standard `NaN` values and domain-specific "implicit" missing values (e.g., bureau score = 0 representing first-time borrowers without credit history).

In [ ]:
# 1. Explicit Missingness (NaN)
missing_counts = df_train.isnull().sum()
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts[missing_counts > 0],
    'Missing_Percentage (%)': (missing_counts[missing_counts > 0] / len(df_train)) * 100
})
print("Explicit Missing Values (NaN):")
display(missing_df if not missing_df.empty else "No explicit NaN values found.")

# 2. Implicit Missingness: Bureau History
ntc_count = (df_train['PERFORM_CNS.SCORE'] == 0).sum()
print(f"\nImplicit Missingness (New to Credit / No Bureau Score):")
print(f"  - Borrowers with 0 Bureau Score: {ntc_count:,} ({ntc_count/len(df_train):.2%})")

### Step 1.5: Feature Taxonomy & Domain Categorization

Before diving into univariate/bivariate plots, an ML Engineer groups the raw columns into functional banking domains.

In [ ]:
# Map columns into business domains
feature_groups = {
    'Identifiers': ['UniqueID', 'branch_id', 'supplier_id', 'manufacturer_id', 'Current_pincode_ID', 'State_ID', 'Employee_code_ID'],
    'Loan & Asset': ['disbursed_amount', 'asset_cost', 'ltv'],
    'Borrower Demographics': ['Date.of.Birth', 'Employment.Type'],
    'Loan Disbursal': ['DisbursalDate'],
    'KYC Identity Flags': ['MobileNo_Avl_Flag', 'Aadhar_flag', 'PAN_flag', 'VoterID_flag', 'Driving_flag', 'Passport_flag'],
    'Bureau Score': ['PERFORM_CNS.SCORE', 'PERFORM_CNS.SCORE.DESCRIPTION'],
    'Primary Credit Profile': [col for col in df_train.columns if col.startswith('PRI.')] + ['PRIMARY.INSTAL.AMT'],
    'Secondary Credit Profile': [col for col in df_train.columns if col.startswith('SEC.')] + ['SEC.INSTAL.AMT'],
    'Credit Bureau Behavioral History': ['NEW.ACCTS.IN.LAST.SIX.MONTHS', 'DELINQUENT.ACCTS.IN.LAST.SIX.MONTHS', 'AVERAGE.ACCT.AGE', 'CREDIT.HISTORY.LENGTH', 'NO.OF_INQUIRIES'],
    'Target': ['loan_default']
}

for category, cols in feature_groups.items():
    print(f"{category:35s} : {len(cols):2d} features")

total_mapped = sum(len(cols) for cols in feature_groups.values())
print(f"\nTotal mapped features: {total_mapped} of {df_train.shape[1]}")